# Training prep 

In [1]:
# imports for all modules under src/preprocessing/*.py
from importlib import import_module
from pathlib import Path
import sys

# Make project src importable from this notebook location
project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Auto-import every preprocessing module and expose as globals
preprocessing_modules = {}
preprocessing_dir = src_path / "preprocessing"
for module_file in sorted(preprocessing_dir.glob("*.py")):
    if module_file.stem.startswith("_"):
        continue
    module = import_module(f"preprocessing.{module_file.stem}")
    preprocessing_modules[module_file.stem] = module
    globals()[module_file.stem] = module

print("===Loaded preprocessing modules:", ", ".join(preprocessing_modules.keys()))

===Loaded preprocessing modules: LowDoc, RevLineCr, accept, approvalDate, approvalFY, balanceGross, base_cleaning, city_bank, createJob, disbursementDate, disbursementGross, example, franchise_code, newExists, noemp, retainedJob, urban_rural


In [2]:
import pandas as pd
from pathlib import Path
import importlib

# Load df only if it is not already in memory
if "df" not in globals():
    project_root = Path.cwd().resolve().parent
    df = pd.read_csv(project_root / "data" / "train.csv")

df.head()

,id,LoanNr_ChkDgt,Name,City,State,Bank,BankState,ApprovalDate,ApprovalFY,NoEmp,...,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept
0,64afe857c28,9448323000,MIDWEST CRANKSHAFT & ENGINE,HARVEY,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,9-Aug-96,1996,28,...,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0
1,1705a7346c2,2854405007,"Iredesign, Limited",CHICAGO,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,10-Dec-07,2008,1,...,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1
2,7439801ad8a,9300423010,PHILLY'S INC.,ROCHELLE,IL,BMO HARRIS BK NATL ASSOC,IL,23-May-96,1996,6,...,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1
3,a3f8f9d0611,4349265000,USA Laser Imaging Inc.,Loves park,IL,ALPINE BANK & TRUST CO.,IL,4-Nov-10,2011,5,...,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1
4,71e4f243b5d,2433905006,"Dan Morrell, Inc.",LISLE,IL,JPMORGAN CHASE BANK NATL ASSOC,IL,3-May-07,2007,3,...,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0


### Base Cleanup
* remove: '`id`', '`LoanNr_ChkDgt`', '`Name`'
* `City` cleanup
* `Bank` cleanup
* created `IsLocalBank`
* Removed `State`
* Removed `BankState`

In [3]:
base_cleaning = importlib.reload(base_cleaning)
df = base_cleaning.clean_base_columns(df)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,28,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,1,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,6,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,5,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,3,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1


###  NoEmp 
* "raw" -> en caso de árboles.( does nothing)
* "log" -> En caso de KNN o SVM for a  logarithmic transformation of NoEmp using log1p
* "binning" -> (creates bins)Para análisis interpretativo, probar en arboles también para ver si mejora o empeora el rendimiento.


* creates `NoEmp_Log` created with log, removes `NoEmp`

* creates `NoEmp_Bin` with binning, removes `NoEmp`

In [4]:
# noemp_option = "raw"   # o "log" o "binning"
noemp_option = "log"
# noemp_option = "binning"

noemp = importlib.reload(noemp)
df = noemp.preprocess_noemp(df, option=noemp_option)
df.head()

,City,Bank,ApprovalDate,ApprovalFY,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,HARVEY,JPMORGAN CHASE BANK NATL ASSOC,9-Aug-96,1996,1.0,0,0,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,CHICAGO,JPMORGAN CHASE BANK NATL ASSOC,10-Dec-07,2008,2.0,1,1,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,ROCHELLE,BMO HARRIS BK NATL ASSOC,23-May-96,1996,2.0,0,0,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,LOVES PARK,ALPINE BANK & TRUST CO.,4-Nov-10,2011,2.0,0,5,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,LISLE,JPMORGAN CHASE BANK NATL ASSOC,3-May-07,2007,1.0,1,3,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### City y Bank
binary encoding <br>
* remove `city` column
* remove `bank` column

In [5]:
city_bank = importlib.reload(city_bank)

df = city_bank.get_city_bank_encoder(df)

print("Current df features:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

df.head()



Current df features:
1. City_0
2. City_1
3. City_2
4. City_3
5. City_4
6. City_5
7. City_6
8. City_7
9. City_8
10. City_9
11. City_10
12. Bank_0
13. Bank_1
14. Bank_2
15. Bank_3
16. Bank_4
17. Bank_5
18. Bank_6
19. Bank_7
20. Bank_8
21. ApprovalDate
22. ApprovalFY
23. NewExist
24. CreateJob
25. RetainedJob
26. FranchiseCode
27. UrbanRural
28. RevLineCr
29. LowDoc
30. DisbursementDate
31. DisbursementGross
32. BalanceGross
33. Accept
34. IsLocalBank
35. NoEmp_Log


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log
0,0,0,0,0,0,0,0,0,0,0,...,1,0,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296
1,0,0,0,0,0,0,0,0,0,1,...,0,1,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147
2,0,0,0,0,0,0,0,0,0,1,...,1,0,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910
3,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759
4,0,0,0,0,0,0,0,0,1,0,...,0,1,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294


### NewExist
* create `is_new_business`
* create `newexist_missing_or_invalid` (option A)
* remove `NewExist`

In [6]:
# calling src\preprocessing\newExists.py

# Reload module to pick up latest code changes
newExists = importlib.reload(newExists)

# Choose NewExist preprocessing path: 'A' or 'B'
newexist_option = "A" #===> is_new_business +  newexist_missing_or_invalid
# Option A: Create 'is_new_business' and 'newexist_missing_or_invalid' columns
# Option B: Create only 'is_new_business' column, and delete rows with missing/invalid values

print(f"NewExist option used: {newexist_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = newExists.preprocess_newexist(
    df=df,
    option=newexist_option,
    source_col="NewExist",
)


print(f"Rows after: {len(df):,}")
df.head()

NewExist option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,RevLineCr,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid
0,0,0,0,0,0,0,0,0,0,0,...,N,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0
1,0,0,0,0,0,0,0,0,0,1,...,N,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0
2,0,0,0,0,0,0,0,0,0,1,...,N,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0
3,0,0,0,0,0,0,0,0,1,0,...,N,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0
4,0,0,0,0,0,0,0,0,1,0,...,N,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0


### CreateJob
* adds `createjob_normalized`  (option A)
* adds `createjob_standardized`  (option B)
* removes `CreateJob` column 


In [7]:
# calling src\preprocessing\createJob.py

# Reload module to pick up latest code changes
createJob = importlib.reload(createJob)

# Choose CreateJob preprocessing path: 'A' (normalize) or 'B' (standardize)
createjob_option = "A"

print(f"CreateJob option used: {createjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = createJob.preprocess_createjob(
    df=df,
    option=createjob_option,
    source_col="CreateJob",
)


print(f"Rows after: {len(df):,}")
df.head()

CreateJob option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,LowDoc,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,N,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0
1,0,0,0,0,0,0,0,0,0,1,...,N,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,Y,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0
3,0,0,0,0,0,0,0,0,1,0,...,N,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0
4,0,0,0,0,0,0,0,0,1,0,...,N,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114


### RetainedJob
* adds `retainedjob_normalized`  (option A)
* adds `retainedjob_standardized`  (option B)
* removes `RetainedJob` column 

In [8]:
# calling src\preprocessing\retainedJob.py

# Reload module to pick up latest code changes
retainedJob = importlib.reload(retainedJob)

# Choose RetainedJob preprocessing path: 'A' (normalize) or 'B' (standardize)
retainedjob_option = "A"
# retainedjob_option = "B"

print(f"RetainedJob option used: {retainedjob_option}")
print(f"Rows before: {len(df):,}")

# Apply preprocessing from the imported module
df = retainedJob.preprocess_retainedjob(
    df=df,
    option=retainedjob_option,
    source_col="RetainedJob",
)


print(f"Rows after: {len(df):,}")
df.head()

RetainedJob option used: A
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,DisbursementDate,DisbursementGross,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized
0,0,0,0,0,0,0,0,0,0,0,...,31-Mar-97,"$600,000.00",$0.00,0,1,3.367296,0,0,0.0,0.0
1,0,0,0,0,0,0,0,0,0,1,...,31-Dec-07,"$25,400.00",$0.00,1,1,0.693147,1,0,0.000114,0.000114
2,0,0,0,0,0,0,0,0,0,1,...,30-Sep-96,"$20,000.00",$0.00,1,1,1.945910,1,0,0.0,0.0
3,0,0,0,0,0,0,0,0,1,0,...,1-Mar-11,"$75,000.00",$0.00,1,1,1.791759,1,0,0.0,0.000568
4,0,0,0,0,0,0,0,0,1,0,...,31-May-07,"$50,000.00",$0.00,0,1,1.386294,0,0,0.000114,0.000341


### ApprovalDate
Procesa la fecha en la que se aprobó el préstamo.
* Opción A (Recomendada para modelos geométricos/numéricos): Extrae el `ApprovalYear` (Año) y `ApprovalMonth` (Mes). Rellena los valores faltantes usando la moda (el valor más común) y aplica Normalización (MinMaxScaler) para que los valores queden en una escala de 0 a 1. Elimina la columna original.
* Opción B (Para exploración): Convierte la columna de texto original a formato datetime y la mantiene en el DataFrame `ApprovalDate` sin extraer componentes.



In [9]:
# calling src\preprocessing\approvalDate.py
from preprocessing import approvalDate
import importlib

# Recargar el módulo para aplicar los últimos cambios en el código
approvalDate = importlib.reload(approvalDate)

# Elegir la ruta de preprocesamiento: 'A' (Extraer y Normalizar) o 'B' (Mantener datetime)
approvaldate_option = "A"

print(f"ApprovalDate option used: {approvaldate_option}")
print(f"Rows before: {len(df):,}")

# Aplicar el preprocesamiento y SOBRESCRIBIR df (para mantener el pipeline)
df = approvalDate.preprocess_approvaldate(
    df=df,
    option=approvaldate_option,
    source_col="ApprovalDate",
)

print(f"Rows after: {len(df):,}")

# Comprobación rápida de las columnas generadas
if approvaldate_option == "A":
    cols_to_show = ["approvalyear_normalized", "approvalmonth_normalized"]
    print("\nOpción A seleccionada: Se crearon las columnas normalizadas (escala 0 a 1).")
    display(df[cols_to_show].head(10))
elif approvaldate_option == "B":
    cols_to_show = ["ApprovalDate"]
    print("\nOpción B seleccionada: Se mantuvo la fecha en formato datetime.")
    display(df[cols_to_show].head(10))

ApprovalDate option used: A
Rows before: 20,768
Rows after: 20,768

Opción A seleccionada: Se crearon las columnas normalizadas (escala 0 a 1).


,approvalyear_normalized,approvalmonth_normalized
0,0.590909,0.636364
1,0.840909,1.000000
2,0.590909,0.363636
3,0.909091,0.909091
4,0.840909,0.363636
5,0.750000,0.181818
6,0.795455,0.818182
7,0.522727,0.090909
8,0.659091,0.363636
9,0.772727,0.181818


### ApprovalFY
Procesa el Año Fiscal del préstamo.
* Opción A: 
Removes the ApprovalFY. El equipo determinó que esta información es redundante si ya tenemos la fecha de aprobación (`ApprovalDate`).

* Opción B: Limpia la columna (elimina letras como 'A'), rellena los valores nulos con la moda, y aplica Normalización (MinMaxScaler) para dejar los años en una escala de 0 a 1<br>
creaes `approvalfy_normalized`<br>
removes `ApprovalDate`


In [10]:
# calling src\preprocessing\approvalFY.py
from preprocessing import approvalFY
import importlib

# Recargar el módulo para aplicar los últimos cambios en el código
approvalFY = importlib.reload(approvalFY)

# Elegir la ruta de preprocesamiento: 'A' (Eliminar) o 'B' (Limpiar y Normalizar)
approvalfy_option = "A"

print(f"ApprovalFY option used: {approvalfy_option}")
print(f"Rows before: {len(df):,}")

# Aplicar el preprocesamiento y guardar en el df principal
df = approvalFY.preprocess_approvalfy(
    df=df,
    option=approvalfy_option,
    source_col="ApprovalFY",
)

print(f"Rows after: {len(df):,}")

# Comprobación rápida
if approvalfy_option == "A":
    print("\nOpción A seleccionada: La columna 'ApprovalFY' fue ELIMINADA del dataset.")
    # Mostramos que ya no está imprimiendo las primeras columnas del df
    display(df.head(3))
elif approvalfy_option == "B":
    cols_to_show = ["approvalfy_normalized"]
    print("\nOpción B seleccionada: Se creó la columna normalizada (escala 0 a 1).")
    display(df[cols_to_show].head(10))

ApprovalFY option used: A
Rows before: 20,768
Rows after: 20,768

Opción A seleccionada: La columna 'ApprovalFY' fue ELIMINADA del dataset.


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,BalanceGross,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized
0,0,0,0,0,0,0,0,0,0,0,...,$0.00,0,1,3.367296,0,0,0.0,0.0,0.590909,0.636364
1,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,0.693147,1,0,0.000114,0.000114,0.840909,1.000000
2,0,0,0,0,0,0,0,0,0,1,...,$0.00,1,1,1.945910,1,0,0.0,0.0,0.590909,0.363636


### FranchiseCode
- 'binary': Convierte a 0 (No franquicia: códigos 0 o 1) y 1 (Sí franquicia: > 1).
- 'raw': Devuelve la columna original sin cambios (rellenando nulos con 0).

* created `IsFranchise`
* remove `FranchiseCode`
    

In [11]:
# # llamando a src\preprocessing\franchise_code.py

franchise_code = importlib.reload(franchise_code)

# Elegir la ruta de preprocesamiento de FranchiseCode: 'binary' o 'raw'
franchise_option = "binary" 

print(f"Opción de FranchiseCode utilizada: {franchise_option}")
print(f"Filas antes: {len(df):,}")

# Aplicar el preprocesamiento desde el módulo importado
df = franchise_code.preprocess_franchise_code(
    df=df,
    option=franchise_option,
    source_col="FranchiseCode",
)


print(f"Filas después: {len(df):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "FranchiseCode" in df.columns:
    cols_to_show.append("FranchiseCode")
if "IsFranchise" in df.columns:
    cols_to_show.append("IsFranchise")

display(df[cols_to_show].head(10))
df.head()


Opción de FranchiseCode utilizada: binary
Filas antes: 20,768
Filas después: 20,768


,IsFranchise
0,0
1,0
2,0
3,0
4,0
5,1
6,0
7,1
8,0
9,0


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,Accept,IsLocalBank,NoEmp_Log,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized,IsFranchise
0,0,0,0,0,0,0,0,0,0,0,...,0,1,3.367296,0,0,0.0,0.0,0.590909,0.636364,0
1,0,0,0,0,0,0,0,0,0,1,...,1,1,0.693147,1,0,0.000114,0.000114,0.840909,1.000000,0
2,0,0,0,0,0,0,0,0,0,1,...,1,1,1.945910,1,0,0.0,0.0,0.590909,0.363636,0
3,0,0,0,0,0,0,0,0,1,0,...,1,1,1.791759,1,0,0.0,0.000568,0.909091,0.909091,0
4,0,0,0,0,0,0,0,0,1,0,...,0,1,1.386294,0,0,0.000114,0.000341,0.840909,0.363636,0


### UrbanRural
- 'text': Mapea los números (0, 1, 2) a strings ('Undefined', 'Urban', 'Rural').
- 'onehot': Mapea a texto y aplica One-Hot Encoding creando 3 columnas binarias.

* created `Zone_Rural`
* created `Zone_Undefined`
* created `Zone_Urban`
* remove `UrbanRural`

In [12]:
# llamando a src\preprocessing\urban_rural.py

urban_rural = importlib.reload(urban_rural)

# Elegir la ruta de preprocesamiento de UrbanRural: 'onehot' o 'text'
urbanrural_option = "onehot" 

print(f"Opción de UrbanRural utilizada: {urbanrural_option}")
print(f"Filas antes: {len(df):,}")

# Aplicar el preprocesamiento desde el módulo importado
df = urban_rural.preprocess_urban_rural(
    df=df,
    option=urbanrural_option,
    source_col="UrbanRural",
)

print(f"Filas después: {len(df):,}")

# Comprobación rápida de las columnas generadas
cols_to_show = []
if "UrbanRural" in df.columns:
    cols_to_show.append("UrbanRural")

# Añadir las columnas one-hot generadas si existen
zone_cols = [col for col in df.columns if col.startswith('Zone_')]
cols_to_show.extend(zone_cols)

display(df[cols_to_show].head(10))
df.head()

Opción de UrbanRural utilizada: onehot
Filas antes: 20,768
Filas después: 20,768


,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,1,0
1,0,0,1
2,0,1,0
3,0,0,1
4,0,0,1
5,0,0,1
6,0,0,1
7,0,1,0
8,0,1,0
9,0,0,1


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,is_new_business,newexist_missing_or_invalid,createjob_normalized,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.590909,0.636364,0,0,1,0
1,0,0,0,0,0,0,0,0,0,1,...,1,0,0.000114,0.000114,0.840909,1.000000,0,0,0,1
2,0,0,0,0,0,0,0,0,0,1,...,1,0,0.0,0.0,0.590909,0.363636,0,0,1,0
3,0,0,0,0,0,0,0,0,1,0,...,1,0,0.0,0.000568,0.909091,0.909091,0,0,0,1
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0.000114,0.000341,0.840909,0.363636,0,0,0,1


### RevLineCr
Option A:

* `RevLineCr_clean`
This is the normalized categorical value used as the main feature.
Current buckets in code: Y, N, UNKNOWN, MISSING.

* `revlinecr_is_nonstandard`
Binary quality flag.
It is 1 when RevLineCr_clean is UNKNOWN (so original value was not one of Y, N, or missing).

* `revlinecr_is_missing`
Binary missingness flag.
It is 1 when RevLineCr_clean is MISSING (true null/NaN after cleaning).

Option B:

* `RevLineCr_clean`

Option C:
* `has_RevLineCr` from RevLineCr_clean, "Y" values = 1
* `revlinecr_is_nonstandard`
* `revlinecr_is_missing`
* Removes `RevLineCr`

In [13]:
# calling src\preprocessing\RevLineCr.py
RevLineCr = importlib.reload(RevLineCr)

# Aplicar opción C

revlinecr_option = "C"
print(f"Rows before: {len(df):,}")
print(f"RevLineCr option used: {revlinecr_option}")

df = RevLineCr.preprocess_revlinecr(
    df=df,
    option=revlinecr_option,
    source_col="RevLineCr",
)

print(f"Rows after: {len(df):,}")
df.head()

Rows before: 20,768
RevLineCr option used: C
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,retainedjob_normalized,approvalyear_normalized,approvalmonth_normalized,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban,revlinecr_is_nonstandard,revlinecr_is_missing,has_RevLineCr
0,0,0,0,0,0,0,0,0,0,0,...,0.0,0.590909,0.636364,0,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0.000114,0.840909,1.000000,0,0,0,1,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0.0,0.590909,0.363636,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,...,0.000568,0.909091,0.909091,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0.000341,0.840909,0.363636,0,0,0,1,0,0,0


### LowDoc

Option A:

* `LowDoc_clean`
This is the normalized categorical value used as the main feature. (0, S, C, A, R) → "UNKNOWN"
Current buckets in code: Y, N, UNKNOWN, MISSING.

* `lowdoc_is_nonstandard`
Binary quality flag.
It is 1 when LowDoc_clean is UNKNOWN (so original value was not one of Y, N, or missing).

* `lowdoc_is_missing`
Binary missingness flag.
It is 1 when LowDoc_clean is MISSING (true null/NaN after cleaning).

Option B:

* `RevLineCr_clean`

Option C:
* `is_LowDoc` from RevLineCr_clean, "Y" values = 1
* `lowdoc_is_nonstandard`
* `lowdoc_is_missing`
* Removes `LowDoc_clean`


In [14]:
# calling src\preprocessing\LowDoc.py
LowDoc = importlib.reload(LowDoc)

# Aplicar opción A
lowdoc_option = "C"
print(f"Rows before: {len(df):,}")
print(f"LowDoc option used: {lowdoc_option}")

df = LowDoc.preprocess_lowdoc(
    df=df,
    option=lowdoc_option,
    source_col="LowDoc",
)

print(f"Rows before: {len(df):,}")
print(f"Rows after: {len(df):,}")

df.head()

Rows before: 20,768
LowDoc option used: C
Rows before: 20,768
Rows after: 20,768


,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban,revlinecr_is_nonstandard,revlinecr_is_missing,has_RevLineCr,lowdoc_is_nonstandard,lowdoc_is_missing,is_LowDoc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0


## DisbursementDate

Remove the column

TODO: - **Disbursement_Delay** (Numerical): `DisbursementDate - ApprovalDate` (in days).
  - **Logic:** Long delays might indicate administrative hurdles or a change in the borrower's financial health during the underwriting process.

In [15]:
# calling src\preprocessing\disbursementDate.py
disbursementDate = importlib.reload(disbursementDate)

df = disbursementDate.preprocess_disbursementdate(
    df=df,
    source_col="DisbursementDate",
)

df.head()

,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban,revlinecr_is_nonstandard,revlinecr_is_missing,has_RevLineCr,lowdoc_is_nonstandard,lowdoc_is_missing,is_LowDoc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0


## BalanceGross

Removed

In [16]:
balanceGross = importlib.reload(balanceGross)

df = balanceGross.preprocess_balancegross(
    df=df,
    source_col="BalanceGross",
)

df.head()

,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban,revlinecr_is_nonstandard,revlinecr_is_missing,has_RevLineCr,lowdoc_is_nonstandard,lowdoc_is_missing,is_LowDoc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0


## Accept

Converted to numeric

In [17]:
accept = importlib.reload(accept)

df = accept.preprocess_accept(
    df=df,
    source_col="Accept",
)

df.head()

,City_0,City_1,City_2,City_3,City_4,City_5,City_6,City_7,City_8,City_9,...,IsFranchise,Zone_Rural,Zone_Undefined,Zone_Urban,revlinecr_is_nonstandard,revlinecr_is_missing,has_RevLineCr,lowdoc_is_nonstandard,lowdoc_is_missing,is_LowDoc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0


## DisbursementGross

* Option A: normalize values (min-max)

* Option B: standardize values (z-score)

In [18]:
# calling src\preprocessing\disbursementGross.py

disbursementGross = importlib.reload(disbursementGross)

# disbursementgross_option = "A" # normalize values (min-max scaling)
disbursementgross_option = "B" # standardize values (z-score)

print(f"DisbursementGross option used: {disbursementgross_option}")
print(f"Rows before: {len(df):,}")

df = disbursementGross.preprocess_disbursementgross(
    df=df,
    option=disbursementgross_option,
    source_col="DisbursementGross",
)

print(f"Rows after: {len(df):,}")

cols_to_show = ["DisbursementGross"]
if "disbursementgross_normalized" in df.columns:
    cols_to_show.append("disbursementgross_normalized")

display(df[cols_to_show].head(10))



DisbursementGross option used: B
Rows before: 20,768
Rows after: 20,768


,DisbursementGross
0,1.469113
1,-0.572057
2,-0.59124
3,-0.395861
4,-0.48467
5,-0.243111
6,-0.48467
7,-0.129436
8,2.890045
9,-0.542573


In [19]:
# Print all features with their pandas dtype
feature_types = df.dtypes.astype(str).reset_index()
feature_types.columns = ["Feature", "Type"]
print(f"Total features: {len(feature_types)}")
display(feature_types)

Total features: 40


,Feature,Type
0,City_0,int64
1,City_1,int64
2,City_2,int64
3,City_3,int64
4,City_4,int64
5,City_5,int64
6,City_6,int64
7,City_7,int64
8,City_8,int64
9,City_9,int64
